In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# This might be needed in order to get plots to display in the exported HTML for submission.
# In any case, please double check that plots display properly before you submit.
import plotly.io as pio
pio.renderers.default='notebook'
df = pd.read_csv("Daily_Port_Activity_Data_and_Trade_Estimates.csv")

In [ ]:
df.columns

In [ ]:
df_imp_exp = df.groupby("portid").agg(
        avg_import=("import", "mean"),
        avg_export=("export", "mean"),
        country=("country", "first"),
        portname=("portname", "first")
    ).reset_index()

px.scatter(df_imp_exp,
           x="avg_export",
           y="avg_import",
           log_x=True,
           log_y=True,
           hover_name="portname",
           hover_data="country")

In [ ]:
df_imp_exp_coutry = df.groupby("ISO3").agg(
        avg_import=("import", "mean"),
        avg_export=("export", "mean"),
        country=("country", "first")
        ).reset_index()

df_imp_exp_coutry["avg_balance"] = df_imp_exp_coutry["avg_export"] - df_imp_exp_coutry["avg_import"]

px.choropleth(
    df_imp_exp_coutry,
    locations="ISO3",
    locationmode="ISO-3",
    color="avg_balance",
    hover_name="country",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    labels={"avg_balance" : "Bilancio medio (tonnelate)"},
    title="Bilancio commerciale medio per paese (export − import)",
    projection="natural earth"
)

In [ ]:
df_time = df.groupby(["year", "month"])[['portcalls_container', 'portcalls_dry_bulk', 'portcalls_general_cargo', 'portcalls_roro', 'portcalls_tanker', 'portcalls_cargo']].sum().reset_index()
df_time["date"] = pd.to_datetime(df_time[["year","month"]].assign(day=1))
px.line(df_time, x="date", y=['portcalls_container', 'portcalls_dry_bulk', 'portcalls_general_cargo', 'portcalls_roro', 'portcalls_tanker', 'portcalls_cargo'], title="Traffico globale mensile")

In [ ]:
import_cols = ["import_container", "import_dry_bulk", "import_general_cargo", "import_roro", "import_tanker"]
export_cols = ["export_container", "export_dry_bulk", "export_general_cargo", "export_roro", "export_tanker"]

df_cargo_country = df.groupby("country")[import_cols + export_cols].mean().reset_index()

# Top 20 per import totale
df_cargo_country["total_import"] = df_cargo_country[import_cols].sum(axis=1)
top20_imp = df_cargo_country.nlargest(5, "total_import")

px.bar(
    top20_imp.melt(id_vars="country", value_vars=import_cols),
    x="country", y="value", color="variable",
    barmode="stack",
    title="Composizione import per paese (top 5)"
)


In [ ]:

# Top 20 per export totale
df_cargo_country["total_export"] = df_cargo_country[export_cols].sum(axis=1)
top20_exp = df_cargo_country.nlargest(5, "total_export")

px.bar(
    top20_exp.melt(id_vars="country", value_vars=export_cols),
    x="country", y="value", color="variable",
    barmode="stack",
    title="Composizione export per paese (top 5)"
)

In [ ]:
# Globale
df_global = df.groupby(["year", "month"])["portcalls"].sum().reset_index()
pivot_global = df_global.pivot(index="year", columns="month", values="portcalls")

px.imshow(
    pivot_global,
    title="Stagionalità import globale (mese × anno)",
    labels={"x": "Mese", "y": "Anno", "color": "Navi transitate"},
    color_continuous_scale="Blues"
)

In [ ]:
# Rotterdam
df_rot = df[df["portname"] == "Rotterdam"].groupby(["year", "month"])["portcalls"].sum().reset_index()
pivot_rot = df_rot.pivot(index="year", columns="month", values="portcalls")

px.imshow(
    pivot_rot,
    title="Stagionalità import - Rotterdam (mese × anno)",
    labels={"x": "Mese", "y": "Anno", "color": "Navi transitate"},
    color_continuous_scale="Blues"
)

In [ ]:
portcalls_cols = ["portcalls_container", "portcalls_dry_bulk", "portcalls_general_cargo", "portcalls_roro", "portcalls_tanker"]

df_annual = df.groupby("year")[portcalls_cols].sum().reset_index()
means = df_annual[portcalls_cols].mean().reset_index()
means.columns = ["tipo", "media"]

px.bar(means.sort_values("media"), x="media", y="tipo", orientation="h",
       title="Distribuzione media annuale del traffico navale per tipo")